# Lunar Pathloss Radio Map — Starter Notebook

The First Lunar Pathloss Radio Map Prediction Challenge (ICASSP 2027 Grand Challenge).

This notebook takes you end to end: read the data, understand how it is packed,
train a small baseline, and write a submission file in the exact format the
leaderboard expects.


In [ ]:
import os, csv, json, math, time
import numpy as np
import matplotlib.pyplot as plt

# Works unchanged on Kaggle and on a local clone. On Kaggle the competition
# directory is whichever /kaggle/input child holds metadata.json, so this keeps
# working if the slug ever changes.
import glob

def find_data():
    if os.environ.get("LUNAR_DATA"):
        return os.environ["LUNAR_DATA"]
    for p in sorted(glob.glob("/kaggle/input/*")) + ["./data", "../data", "."]:
        if os.path.exists(os.path.join(p, "metadata.json")):
            return p
    return None

DATA = find_data()
assert DATA, ("dataset not found. On Kaggle, add the competition data to this "
              "notebook. Locally, set LUNAR_DATA=/path/to/unpacked/data.")
print("data root:", DATA)
for f in sorted(os.listdir(DATA)):
    size = os.path.getsize(os.path.join(DATA, f))
    print(f"  {f:24} {size/1e6:10.1f} MB")

## How the data is packed

The data is **consolidated arrays**, one file per split per field, not one file
per sample. Row *i* of `train_pl415.npy` is row *i* of `train_index.csv`. That
ordering is the whole contract.

Three things are compressed out and have to be rebuilt on load:

| What | Why | How to get it back |
|---|---|---|
| Heightmaps are deduplicated | a terrain with 51 transmitters has 51 samples sharing one heightmap | `hm[int(row["hm_row"])]` |
| TX maps are not stored | a one-hot 256×256 array is two integers | `tx[row["tx_row"], row["tx_col"]] = 1` |
| Masks are bit-packed | 8× smaller, and exact | `np.unpackbits(packed).reshape(256, 256)` |

Always open the big arrays with `mmap_mode="r"`. `train_pl415.npy` is 3.4 GB;
memory-mapping is the difference between working and dying.

In [ ]:
meta = json.load(open(os.path.join(DATA, "metadata.json")))
PL_MIN, PL_MAX = meta["pathloss_range_db"]
HM_MIN, HM_MAX = meta["heightmap_range_m"]
print("pathloss range (dB):", PL_MIN, PL_MAX, " scale:", PL_MAX - PL_MIN)
print("heightmap range (m):", HM_MIN, HM_MAX)
print("splits:", json.dumps(meta["splits"], indent=2))

def load_index(split):
    with open(os.path.join(DATA, f"{split}_index.csv"), newline="") as f:
        return list(csv.DictReader(f))

train_rows, val_rows = load_index("train"), load_index("val")
print(f"\ntrain {len(train_rows):,} samples/band, val {len(val_rows):,} samples/band")
print("index columns:", list(train_rows[0]))

## Unpacking one sample by hand

Do this once so the shapes are not mysterious later.

In [ ]:
def arr(split, field):
    return np.load(os.path.join(DATA, f"{split}_{field}.npy"), mmap_mode="r")

row = val_rows[0]
hm = np.asarray(arr("val", "hm")[int(row["hm_row"])]).astype(np.float32)

tx = np.zeros((256, 256), np.float32)
tx[int(row["tx_row"]), int(row["tx_col"])] = 1.0

pl415 = np.asarray(arr("val", "pl415")[0]).astype(np.float32)
m415 = np.unpackbits(np.asarray(arr("val", "mask415")[0])).reshape(256, 256)
pl58 = np.asarray(arr("val", "pl58")[0]).astype(np.float32)
m58 = np.unpackbits(np.asarray(arr("val", "mask58")[0])).reshape(256, 256)

print("sample", row["sample_id"], "terrain", row["terrain_id"])
print(f"heightmap {hm.shape} {hm.dtype}  [{hm.min():.1f}, {hm.max():.1f}] m")
print(f"tx one-hot sums to {tx.sum():.0f} at ({row['tx_row']}, {row['tx_col']})")
print(f"415 MHz  valid {m415.mean():6.1%}   5.8 GHz  valid {m58.mean():6.1%}")

fig, ax = plt.subplots(1, 5, figsize=(19, 3.6))
for a, img, title in zip(
        ax,
        [hm, pl415, m415, pl58, m58],
        ["heightmap (m)", "415 MHz (dB)", "415 MHz mask", "5.8 GHz (dB)", "5.8 GHz mask"]):
    im = a.imshow(img, cmap="terrain" if "heightmap" in title else "viridis" if "mask" not in title else "gray")
    a.set_title(title); a.axis("off")
    fig.colorbar(im, ax=a, fraction=0.046)
ax[1].scatter([int(row["tx_col"])], [int(row["tx_row"])], c="red", s=40, marker="x")
plt.tight_layout(); plt.show()

Notice how much of the 5.8 GHz map is invalid. That is the ray tracer failing to
resolve deep shadow, and those pixels are excluded from scoring. The voids are
geometrically structured, which is exactly why an all-pixel loss is tempting and
wrong: a model can score well by learning where the tracer failed instead of
learning propagation.

## The reference dataloader

`lunar_dataset.py` ships with the data and does all of the above for you. It
auto-detects the layout, memory-maps the arrays, rebuilds the TX one-hot and
unpacks the masks.

In [ ]:
import sys
sys.path.insert(0, DATA)
from lunar_dataset import LunarRadioMapDataset

import torch
from torch.utils.data import DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Windows spawns dataloader workers, which needs a __main__ guard a notebook
# does not have; Kaggle is Linux, so this is 2 there and 0 on a local Windows run.
NUM_WORKERS = 0 if os.name == "nt" else 2
BAND = "both"         # "both" = one conditional model over both bands.
                      # Set "415" or "58" to train a single-band model --
                      # but the ranking is the 50/50 mean of BOTH bands,
                      # so a single-band model cannot place.
print("device:", DEVICE, "| torch", torch.__version__)

train_ds = LunarRadioMapDataset(DATA, split="train", band=BAND,
                                return_mask=True, augment=True)
val_ds = LunarRadioMapDataset(DATA, split="val", band=BAND, return_mask=True)
print(f"layout {train_ds.layout} | train {len(train_ds):,} | val {len(val_ds):,}")

x, y, m = train_ds[0]
print("x", tuple(x.shape), "y", tuple(y.shape), "mask", tuple(m.shape))
print("inputs are normalized to [0,1]; denormalize with train_ds.denormalize_pathloss(y)")

## A small baseline

A compact U-Net, defined inline so this notebook stands alone. The **full
RadioUNet baseline**, the one whose numbers appear on the challenge site, lives
in the challenge repository along with `train.py` and `evaluate.py`.

In [ ]:
import torch.nn as nn

def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class SmallUNet(nn.Module):
    def __init__(self, in_ch, width=32):
        super().__init__()
        w = width
        self.e1, self.e2, self.e3 = block(in_ch, w), block(w, w*2), block(w*2, w*4)
        self.bott = block(w*4, w*8)
        self.u3 = nn.ConvTranspose2d(w*8, w*4, 2, 2); self.d3 = block(w*8, w*4)
        self.u2 = nn.ConvTranspose2d(w*4, w*2, 2, 2); self.d2 = block(w*4, w*2)
        self.u1 = nn.ConvTranspose2d(w*2, w, 2, 2);   self.d1 = block(w*2, w)
        self.out = nn.Conv2d(w, 1, 1)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1)); e3 = self.e3(self.pool(e2))
        b = self.bott(self.pool(e3))
        d3 = self.d3(torch.cat([self.u3(b), e3], 1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], 1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], 1))
        return self.out(d1)

IN_CH = 3 if BAND == "both" else 2
model = SmallUNet(IN_CH).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

## Masked loss

This is the part that matters. `m` is 1 on ray-traced pixels and 0 on fill, so
the loss is a mean over valid pixels only, and it should be normalized by
`m.sum()`, not by the number of elements, or maps with large voids are silently
down-weighted.

In [ ]:
def masked_mse(pred, y, m):
    se = (pred - y) ** 2 * m
    denom = m.sum()
    return se.sum() / denom if denom > 0 else se.sum() * 0.0

SCALE = PL_MAX - PL_MIN   # 208.0 dB; normalized error x SCALE = error in dB

def masked_rmse_db(pred, y, m):
    return math.sqrt(float(masked_mse(pred, y, m))) * SCALE

## Train

`STEPS` is deliberately small so the notebook finishes quickly. Raise it (and
switch to a GPU) for anything you would actually submit.

In [ ]:
STEPS = 200
BATCH = 8

loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=(DEVICE == "cuda"), drop_last=True)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

model.train()
t0, step = time.time(), 0
while step < STEPS:
    for x, y, m in loader:
        x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
        loss = masked_mse(model(x), y, m)
        opt.zero_grad(); loss.backward(); opt.step()
        step += 1
        if step % 50 == 0 or step == 1:
            print(f"step {step:4d}/{STEPS}  loss {loss.item():.5f}  "
                  f"(~{math.sqrt(loss.item())*SCALE:.2f} dB)  {time.time()-t0:.0f}s")
        if step >= STEPS:
            break
print("done in", f"{time.time()-t0:.0f}s")

## Validate in dB

Report the masked RMSE in dB, the same quantity the challenge ranks on.

In [ ]:
@torch.no_grad()
def evaluate(ds, n_batches=20, batch=8):
    model.eval()
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=NUM_WORKERS)
    se = cnt = 0.0
    for i, (x, y, m) in enumerate(dl):
        if i >= n_batches:
            break
        x, y, m = x.to(DEVICE), y.to(DEVICE), m.to(DEVICE)
        pred = model(x)
        se += float((((pred - y) ** 2) * m).sum())
        cnt += float(m.sum())
    return math.sqrt(se / cnt) * SCALE

print(f"val masked RMSE: {evaluate(val_ds):.3f} dB")

## Look at what it predicted

SHow predicted maps

In [ ]:
def _cmap(name, bad="0.85"):
    """Colormap that paints masked-out pixels flat grey instead of leaving holes."""
    c = plt.get_cmap(name).copy()
    c.set_bad(bad)
    return c


@torch.no_grad()
def show_prediction(idx, ds=None):
    """Truth vs. prediction vs. error for one sample. Returns its masked RMSE."""
    ds = val_ds if ds is None else ds
    model.eval()
    x, y, m = ds[idx]
    pred = model(x[None].to(DEVICE))[0, 0].cpu().numpy()

    gt = y[0].numpy() * SCALE + PL_MIN            # normalized [0,1] -> dB
    pr = pred * SCALE + PL_MIN
    keep = m[0].numpy().astype(bool)
    rmse = float(np.sqrt(np.mean((pr - gt)[keep] ** 2)))

    # Hide the gap-filled pixels. They are a 145.0 dB placeholder at 5.8 GHz,
    # not physics, and they are excluded from scoring -- so showing them would
    # only make the model look worse (or better) than it is judged to be.
    show_gt = np.where(keep, gt, np.nan)
    show_pr = np.where(keep, pr, np.nan)
    err = np.where(keep, pr - gt, np.nan)

    # .rows / .samples rather than a helper method: the loader being imported is
    # the one shipped with the data, which may predate any newer convenience.
    i_row, band = ds.samples[idx]
    name = ds.rows[i_row]["sample_id"]

    hm = x[0].numpy() * (HM_MAX - HM_MIN) + HM_MIN
    tr, tc = np.unravel_index(int(np.argmax(x[1].numpy())), (256, 256))

    # Truth and prediction must share a colour scale or the comparison lies.
    lo, hi = np.nanpercentile(show_gt, [1, 99])
    span = float(np.nanmax(np.abs(err)))

    panels = [
        (hm,      "terrain (m)",                   dict(cmap=_cmap("terrain"))),
        (show_gt, "ground truth (dB)",             dict(cmap=_cmap("viridis"), vmin=lo, vmax=hi)),
        (show_pr, "prediction (dB)",               dict(cmap=_cmap("viridis"), vmin=lo, vmax=hi)),
        (err,     f"error (dB)   RMSE {rmse:.2f}", dict(cmap=_cmap("coolwarm"), vmin=-span, vmax=span)),
    ]

    fig, ax = plt.subplots(1, 4, figsize=(17, 3.8))
    for a, (img, title, kw) in zip(ax, panels):
        im = a.imshow(img, **kw)
        a.scatter([tc], [tr], c="red", s=32, marker="x", linewidths=1.4)
        a.set_title(title, fontsize=10)
        a.axis("off")
        fig.colorbar(im, ax=a, fraction=0.046)
    fig.suptitle(f"{name} @ {'415 MHz' if band == '415' else '5.8 GHz'}"
                 f"   -   {keep.mean():.1%} of pixels valid   -   grey = excluded",
                 fontsize=11, y=1.06)
    plt.tight_layout()
    plt.show()
    return rmse


# One sample per band when the model is conditional; otherwise two of the same.
if BAND == "both":
    picks = [next(i for i in range(len(val_ds)) if val_ds.samples[i][1] == b)
             for b in ("415", "58")]
else:
    picks = [0, 1]

for i in picks:
    show_prediction(i)

# show_prediction(idx) takes any index into val_ds -- go hunt for the ones your
# model handles badly, they are more informative than the average case.

## Write a submission

The leaderboard scores **one row per pixel**. `sample_submission.csv` lists every
`ID` you must supply:

```
ID,PL
pl415_00047_00_3,0
pl415_00047_00_9,0
```

`ID` is `pl<band>_<sample_id>_<flat_pixel_index>`, and the pixel index is
**row-major** over the 256×256 grid, that is, `arr.reshape(-1)` order. Getting
this transposed is the single most common way to produce a valid-looking file
that scores like noise.

The cell below groups the required IDs by map, predicts each map once, and reads
off the requested pixels.

In [ ]:
sub_path = os.path.join(DATA, "sample_submission.csv")
with open(sub_path, newline="") as f:
    sub_ids = [r["ID"] for r in csv.DictReader(f)]
print(f"{len(sub_ids):,} rows to fill")

# ID -> (band, sample_id, pixel). rsplit twice: sample_id itself contains "_".
wanted = {}
for rid in sub_ids:
    head, px = rid.rsplit("_", 1)
    band = head.split("_", 1)[0][2:]          # "pl415" -> "415"
    sample_id = head.split("_", 1)[1]
    wanted.setdefault((band, sample_id), []).append((int(px), rid))
print(f"{len(wanted)} distinct maps to predict")

val_row_of = {r["sample_id"]: i for i, r in enumerate(val_rows)}
val_hm = arr("val", "hm")

@torch.no_grad()
def predict_map(band, sample_id):
    r = val_rows[val_row_of[sample_id]]
    hm = np.asarray(val_hm[int(r["hm_row"])]).astype(np.float32)
    hm = np.clip((hm - HM_MIN) / (HM_MAX - HM_MIN), 0, 1)
    tx = np.zeros((256, 256), np.float32)
    tx[int(r["tx_row"]), int(r["tx_col"])] = 1.0
    chans = [hm, tx]
    if BAND == "both":
        chans.append(np.full((256, 256), 0.0 if band == "415" else 1.0, np.float32))
    x = torch.from_numpy(np.stack(chans)[None]).to(DEVICE)
    pred = model(x)[0, 0].cpu().numpy()
    return pred * SCALE + PL_MIN          # back to dB

model.eval()
pl_of = {}
for k, (band, sample_id) in enumerate(wanted):
    if BAND != "both" and band != BAND:
        continue
    pl_of[(band, sample_id)] = predict_map(band, sample_id).reshape(-1)
    if (k + 1) % 50 == 0:
        print(f"  predicted {k+1}/{len(wanted)}")
print("predicted", len(pl_of), "maps")

### Fill and check

A model trained on one band cannot predict the other, so any map outside `BAND`
is written as a constant. That is fine for a format check, but if you are
submitting for real, set `BAND = "both"` or train a second model, because those
rows are scored too.

In [ ]:
FALLBACK = float(np.mean([PL_MIN, PL_MAX]))
out_path = "submission.csv"
n_pred = n_fallback = 0

with open(out_path, "w", newline="") as f:
    f.write("ID,PL\n")
    for (band, sample_id), items in wanted.items():
        flat = pl_of.get((band, sample_id))
        for px, rid in items:
            if flat is None:
                v = FALLBACK; n_fallback += 1
            else:
                v = float(flat[px]); n_pred += 1
            f.write(f"{rid},{v:.2f}\n")

print(f"wrote {out_path}: {n_pred:,} predicted, {n_fallback:,} fallback")

# The submission must contain exactly the sample_submission IDs -- no more, no
# fewer. Kaggle rejects the file otherwise, and this catches it in seconds.
with open(out_path, newline="") as f:
    got = [r["ID"] for r in csv.DictReader(f)]
assert len(got) == len(sub_ids), f"row count {len(got)} != {len(sub_ids)}"
assert set(got) == set(sub_ids), "ID set does not match sample_submission.csv"
print("ID set matches sample_submission.csv exactly")

import itertools
with open(out_path) as f:
    print("".join(itertools.islice(f, 4)))

## Where to go next

- **Full baseline.** The challenge repository has the RadioUNet baseline
  (`train.py`, `evaluate.py`, `score_submission.py`) and the vendored upstream
  model code. Its reference numbers are 4.66 dB at 415 MHz and 6.28 dB at
  5.8 GHz, 5.47 dB combined.
- **Train both bands.** The ranking is the 50/50 mean of the two per-band RMSEs,
  so a model for only one band cannot place.
- **Keep the loss masked.** Training on every pixel measurably improves the
  all-pixel diagnostic while making the ranked metric *worse*.
- **December is a different submission.** The final submission is not a CSV: one
  `.npy` per test sample per band, plus your model and inference code. Runtime
  must stay under 500 ms per map. `score_submission.py --validate-only` checks
  your directory before you send it.